# Irodori-TTS Colab

Google Colab で Irodori-TTS の Gradio UI を起動するノートブックです。Runtime > Change runtime type で GPU を選んでから上から実行してください。

長文分割対応版を使うには、`REPO_URL` を変更済みコードが入った自分の fork にしてください。

In [ ]:
# GPU check
!nvidia-smi

In [ ]:
# Settings
REPO_URL = "https://github.com/ikehiro79/Irodori-TTS.git"
BRANCH = "main"

# "normal" or "voicedesign"
APP_MODE = "normal"

# Only needed for private Hugging Face models.
HF_TOKEN = ""

# Set True only when you manually uploaded /content/Irodori-TTS.
USE_EXISTING_REPO = False

In [ ]:
# Clone or refresh repository
from pathlib import Path

repo_dir = Path("/content/Irodori-TTS")

if USE_EXISTING_REPO:
    if not repo_dir.exists():
        raise FileNotFoundError("/content/Irodori-TTS was not found.")
else:
    if repo_dir.exists():
        %cd /content/Irodori-TTS
        !git remote set-url origin {REPO_URL}
        !git fetch origin {BRANCH}
        !git checkout {BRANCH}
        !git reset --hard origin/{BRANCH}
        !git clean -fd
    else:
        %cd /content
        !git clone --branch {BRANCH} {REPO_URL} Irodori-TTS

%cd /content/Irodori-TTS
!git remote -v
!git status --short
!grep -R "Long Text Chunk Chars" -n /content/Irodori-TTS || true

In [ ]:
# Colab / Gradio compatibility patch
# Some Gradio versions do not accept launch(css=...). Move css to Blocks if needed.
from pathlib import Path

for file in ["gradio_app.py", "gradio_app_voicedesign.py"]:
    path = Path(file)
    if not path.exists():
        continue
    text = path.read_text(encoding="utf-8")
    text = text.replace(
        'with gr.Blocks(title="Irodori-TTS Gradio") as demo:',
        'with gr.Blocks(title="Irodori-TTS Gradio", css=EMOJI_PALETTE_CSS) as demo:',
    )
    text = text.replace(
        'with gr.Blocks(title="Irodori-TTS VoiceDesign Gradio") as demo:',
        'with gr.Blocks(title="Irodori-TTS VoiceDesign Gradio", css=EMOJI_PALETTE_CSS) as demo:',
    )
    text = text.replace("        css=EMOJI_PALETTE_CSS,\n", "")
    path.write_text(text, encoding="utf-8")

print("patched Gradio css compatibility")

In [ ]:
# Hugging Face login. Public models do not need this.
if HF_TOKEN.strip():
    !huggingface-cli login --token {HF_TOKEN}
else:
    print("HF_TOKEN is empty; skipping Hugging Face login.")

In [ ]:
# Install dependencies
# sentencepiece<0.2 can fail on Colab Python, so install 0.2.0 first and remove the old constraint.
!python -m pip install -U pip setuptools wheel
!python -m pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!python -m pip install sentencepiece==0.2.0
!grep -v '^sentencepiece' requirements.txt > requirements_colab.txt
!python -m pip install -r requirements_colab.txt

In [ ]:
# Verify install
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# Launch Gradio UI
# Open the https://xxxxx.gradio.live URL printed in the log.
import shlex

if APP_MODE.lower().strip() == "voicedesign":
    app_file = "gradio_app_voicedesign.py"
    port = 7861
else:
    app_file = "gradio_app.py"
    port = 7860

cmd = f"python {shlex.quote(app_file)} --server-name 0.0.0.0 --server-port {port} --share"
print(cmd)
!{cmd}

## Long Text Settings

長文分割対応版では `Advanced (Optional)` に次の項目が出ます。

- `Long Text Chunk Chars (0=off)`: 1チャンクあたりの文字数。デフォルトは `30`。
- `Chunk Silence ms`: チャンク間に入れる無音。デフォルトは `120` ms。

声質の統一感を出すため、分割された全チャンクで同じ seed を使います。長文では `Seconds` は空欄がおすすめです。